# Enterprise NLP Demo: Custom Tokenizers and Lookup

This notebook demonstrates practical customization patterns built on ideas from `01-tokenizers.ipynb`, especially lookup layers/tables.

Goal: show what to customize, why it matters for enterprise apps, and what problem each customization solves.

## 1) Enterprise Problems We Will Solve

Common production issues:
- Noisy input (`Order#12345`, URLs, emails, punctuation-heavy text).
- Domain words not in generic vocabulary (`chargeback`, `SLA-breach`, `k8s`).
- Multi-tenant differences (each client has different vocabulary).
- Fast rule updates needed without retraining full model.
- OOV drift in live traffic causing unstable predictions.

In [ ]:
import re
import os
import tensorflow as tf

print('TensorFlow:', tf.__version__)

## 2) Custom Text Normalization for Enterprise Input

Problem solved:
- Raw enterprise text contains IDs/URLs/noise that explode vocabulary size.

Customization:
- Replace patterns with stable placeholders (`<ORDER_ID>`, `<URL>`).

Result:
- Better vocabulary stability, better generalization, easier monitoring.

In [ ]:
sample_text = tf.constant([
    'Order#12345 failed after payment gateway timeout',
    'Open ticket TKT-9981 and check https://status.example.com',
    'Customer email is user42@example.com and case id is 7777'
])

def enterprise_standardize(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r'https?://\S+', ' <URL> ')
    x = tf.strings.regex_replace(x, r'order#?\d+', ' <ORDER_ID> ')
    x = tf.strings.regex_replace(x, r'tkt-\d+', ' <TICKET_ID> ')
    x = tf.strings.regex_replace(x, r'\b\S+@\S+\.\S+\b', ' <EMAIL> ')
    x = tf.strings.regex_replace(x, r'\b\d+\b', ' <NUM> ')
    x = tf.strings.regex_replace(x, r'[^a-z<>_ ]', ' ')
    x = tf.strings.regex_replace(x, r'\s+', ' ')
    return tf.strings.strip(x)

vectorizer = tf.keras.layers.TextVectorization(
    standardize=enterprise_standardize,
    split='whitespace',
    output_mode='int',
    output_sequence_length=10
)
vectorizer.adapt(sample_text)

print('Vocabulary:', vectorizer.get_vocabulary()[:25])
print('Vectorized:\n', vectorizer(sample_text).numpy())

## 2A) TextVectorization Argument Variants (Using External Functions)

These are separate examples that follow the same pattern as `enterprise_standardize`.

Goal:
- Keep preprocessing logic in reusable external functions.
- Change TextVectorization behavior by swapping function + arguments.
- Compare output for production design decisions.

In [ ]:
demo_text = tf.constant([
    'SLA-breach on Order#9988 in EU-West',
    'Payment latency spiked on https://ops.example.com',
    'Ticket TKT-1234 escalated to L2 support'
])


def standardize_light(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r'[^a-z0-9\- ]', ' ')
    x = tf.strings.regex_replace(x, r'\s+', ' ')
    return tf.strings.strip(x)


def standardize_aggressive(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r'https?://\S+', ' <URL> ')
    x = tf.strings.regex_replace(x, r'order#?\d+', ' <ORDER_ID> ')
    x = tf.strings.regex_replace(x, r'tkt-\d+', ' <TICKET_ID> ')
    x = tf.strings.regex_replace(x, r'\b\d+\b', ' <NUM> ')
    x = tf.strings.regex_replace(x, r'[^a-z<>_ ]', ' ')
    x = tf.strings.regex_replace(x, r'\s+', ' ')
    return tf.strings.strip(x)


def split_on_space_and_hyphen(x):
    x = tf.strings.regex_replace(x, '-', ' ')
    return tf.strings.split(x)


def build_vectorizer(texts, *, standardize_fn, split_fn='whitespace', output_mode='int',
                     ngrams=None, output_sequence_length=10, max_tokens=None):
    layer = tf.keras.layers.TextVectorization(
        standardize=standardize_fn,
        split=split_fn,
        output_mode=output_mode,
        ngrams=ngrams,
        output_sequence_length=output_sequence_length if output_mode == 'int' else None,
        max_tokens=max_tokens
    )
    layer.adapt(texts)
    return layer

### Separate runnable examples

Below are independent examples that each vary TextVectorization arguments using external functions.

Run them one by one to compare behavior.

### Example C: `output_mode` and `ngrams` with external functions

Problem solved:
- Some systems need token ids (`int`) while others need bag-of-words style features (`tf-idf`).

What changes:
- Keep external function style, but vary `output_mode` and add bigrams using `ngrams=(1, 2)`.

In [ ]:
vec_tfidf = build_vectorizer(
    demo_text,
    standardize_fn=standardize_aggressive,
    split_fn='whitespace',
    output_mode='tf-idf',
    ngrams=(1, 2),
    max_tokens=50
)

features = vec_tfidf(demo_text)
print('TF-IDF vocabulary size:', len(vec_tfidf.get_vocabulary()))
print('TF-IDF shape          :', features.shape)
print('First row (rounded)   :', tf.round(features[0] * 1000) / 1000)

### Example B: `split` argument with external split function

Problem solved:
- Domain tokens like `SLA-breach` or `eu-west` can be split differently based on business need.

What changes:
- Compare default whitespace split vs custom `split_on_space_and_hyphen`.

In [ ]:
vec_default_split = build_vectorizer(
    demo_text,
    standardize_fn=standardize_light,
    split_fn='whitespace',
    output_mode='int',
    output_sequence_length=12
)

vec_custom_split = build_vectorizer(
    demo_text,
    standardize_fn=standardize_light,
    split_fn=split_on_space_and_hyphen,
    output_mode='int',
    output_sequence_length=12
)

print('Default split vocab:', vec_default_split.get_vocabulary()[:20])
print('Custom split vocab :', vec_custom_split.get_vocabulary()[:20])
print('Default split vectors:\n', vec_default_split(demo_text).numpy())
print('Custom split vectors:\n', vec_custom_split(demo_text).numpy())

### Example A: `standardize` argument with different external functions

Problem solved:
- Decide how much normalization your domain needs.

What changes:
- Same input, but `standardize_light` vs `standardize_aggressive`.

In [ ]:
vec_light = build_vectorizer(
    demo_text,
    standardize_fn=standardize_light,
    split_fn='whitespace',
    output_mode='int',
    output_sequence_length=12
)

vec_aggr = build_vectorizer(
    demo_text,
    standardize_fn=standardize_aggressive,
    split_fn='whitespace',
    output_mode='int',
    output_sequence_length=12
)

print('Light vocab      :', vec_light.get_vocabulary()[:20])
print('Aggressive vocab :', vec_aggr.get_vocabulary()[:20])
print('Light vectors:\n', vec_light(demo_text).numpy())
print('Aggressive vectors:\n', vec_aggr(demo_text).numpy())

### Quick takeaway

Using external functions keeps preprocessing modular.
In enterprise apps, this makes A/B testing and governance easier because behavior changes are explicit and versionable.

## 3) Domain-Specific `StringLookup` for Stable Label/Token Mapping

Problem solved:
- Different teams use different domain tokens and unknown terms appear frequently.

Customization:
- Build `StringLookup` with domain vocabulary and multiple OOV buckets.

Result:
- Unknown tokens are distributed (hashed) across OOV buckets instead of collapsing to one ID.

In [ ]:
domain_vocab = [
    'payment', 'refund', 'chargeback', 'incident',
    'sla', 'latency', 'outage', 'priority'
]

lookup = tf.keras.layers.StringLookup(
    vocabulary=domain_vocab,
    num_oov_indices=3,
    mask_token=None
)

query_tokens = tf.constant(['refund', 'latency', 'k8s', 'escalation', 'chargeback'])
token_ids = lookup(query_tokens)

print('Query tokens:', query_tokens.numpy())
print('Token ids    :', token_ids.numpy())
print('Vocabulary   :', lookup.get_vocabulary())

## 4) `tf.lookup.StaticHashTable` for Business Rules

Problem solved:
- Need deterministic rule-based mapping (country aliases, channel names, product groups).

Customization:
- Use `KeyValueTensorInitializer` + `StaticHashTable` with fallback.

Result:
- Fast and reproducible mapping in training and serving.

In [ ]:
keys = tf.constant(['us', 'usa', 'united states', 'uk', 'united kingdom', 'in', 'india'])
values = tf.constant(['US', 'US', 'US', 'UK', 'UK', 'IN', 'IN'])

kv_init = tf.lookup.KeyValueTensorInitializer(keys=keys, values=values)
country_table = tf.lookup.StaticHashTable(initializer=kv_init, default_value='UNKNOWN_COUNTRY')

country_query = tf.constant(['usa', 'uk', 'de', 'india'])
mapped = country_table.lookup(country_query)

print('Query :', country_query.numpy())
print('Mapped:', mapped.numpy())

## 5) `TextFileInitializer` for Versioned Vocabulary Artifacts

Problem solved:
- Large vocab should be managed outside code and version-controlled.

Customization:
- Store vocab in file, load with `TextFileInitializer`.

Result:
- Cleaner deployment pipeline, easy rollback by file version.

In [ ]:
lookup_dir = 'tmp_enterprise_lookup'
tf.io.gfile.makedirs(lookup_dir)
vocab_file = os.path.join(lookup_dir, 'intent_vocab.txt')

# One token per line -> id is line number
tf.io.gfile.GFile(vocab_file, 'w').write('billing\ntechnical\naccount\nsecurity\n')

file_init = tf.lookup.TextFileInitializer(
    filename=vocab_file,
    key_dtype=tf.string,
    key_index=tf.lookup.TextFileIndex.WHOLE_LINE,
    value_dtype=tf.int64,
    value_index=tf.lookup.TextFileIndex.LINE_NUMBER
)
intent_table = tf.lookup.StaticHashTable(initializer=file_init, default_value=-1)

intent_query = tf.constant(['security', 'billing', 'legal'])
print('Intent ids:', intent_table.lookup(intent_query).numpy())

## 6) `MutableHashTable` for Hotfix Updates

Problem solved:
- Need emergency additions (new product name, new issue code) before full retrain.

Customization:
- Insert/update keys in mutable table at runtime.

Result:
- Fast adaptation, but requires strict governance for reproducibility.

In [ ]:
mutable = tf.lookup.experimental.MutableHashTable(
    key_dtype=tf.string,
    value_dtype=tf.int64,
    default_value=-1
)

mutable.insert(tf.constant(['outage', 'degraded']), tf.constant([10, 20], dtype=tf.int64))
print('Before update:', mutable.lookup(tf.constant(['outage', 'maintenance'])).numpy())

# Hotfix arrives from operations team
mutable.insert(tf.constant(['maintenance']), tf.constant([30], dtype=tf.int64))
print('After update :', mutable.lookup(tf.constant(['outage', 'maintenance', 'unknown_event'])).numpy())

## 7) Problem -> Customization -> Business Value

| Enterprise Problem | Customization | What It Solves |
|---|---|---|
| Noisy user text (IDs, links, emails) | Custom `standardize` in `TextVectorization` | Reduces vocab explosion and improves model stability |
| Domain-specific terminology | Domain vocabulary in `StringLookup` | Better mapping consistency and lower semantic drift |
| Unknown terms in real traffic | Multiple OOV buckets (`num_oov_indices`) | Better unknown handling than one fallback id |
| Deterministic policy mappings | `StaticHashTable` | Reproducible and auditable rule mapping |
| Vocabulary managed by platform team | `TextFileInitializer` + artifact versioning | Clean CI/CD and easy rollback |
| Urgent runtime updates | `MutableHashTable` | Fast hotfix without immediate full retraining |

### Important Note
For production, version tokenizer config + vocabulary artifact + lookup rules + model together.
That is the minimum contract for reliable enterprise ML deployment.